In [1]:
print("hello world")

hello world


In [2]:
import re
import time
import csv
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

[1. 기본 세팅]
- bash에 입력(selenium 연결)

"/c/Program Files (x86)/Microsoft/Edge/Application/msedge.exe" --remote-debugging-port=9222 --user-data-dir="C:/selenium/edge-profile"

In [5]:
# 정상 작동 확인(실제로 edge가 이동해야 정상)
from selenium import webdriver
from selenium.webdriver.edge.options import Options

options = Options()
options.use_chromium = True
options.debugger_address = "127.0.0.1:9222"

driver = webdriver.Edge(options=options)

print("제목:", driver.title)
print("현재 URL:", driver.current_url)

driver.get("https://viewership.softc.one/ranking/virtualsoftcone?startDateTime=2026-02-28T15%3A00%3A00.000Z&endDateTime=2026-03-31T14%3A59%3A59.999Z")

print("이동 후 제목:", driver.title)
print("이동 후 URL:", driver.current_url)

제목: 소프트콘 뷰어십 SOFTC.ONE VIEWERSHIP : 트위치, 치지직, 유튜브, SOOP 뷰어십, 랭킹
현재 URL: https://viewership.softc.one/
이동 후 제목: 버추얼 소프트콘 랭킹 : 소프트콘 뷰어십 SOFTC.ONE VIEWERSHIP
이동 후 URL: https://viewership.softc.one/ranking/virtualsoftcone?startDateTime=2026-02-28T15%3A00%3A00.000Z&endDateTime=2026-03-31T14%3A59%3A59.999Z


[2. 2026년 3월 1~31일 버추얼 스트리머 랭킹 데이터 수집]
- 스트리머별 통계 데이터를 수집하기 위해서는 랭킹 데이터 안의 고유 채널 id 필요

In [5]:
import csv
import re
import time
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# =========================
# 0. Edge 디버그 연결
# =========================
options = Options()
options.use_chromium = True
options.debugger_address = "127.0.0.1:9222"

driver = webdriver.Edge(options=options)

# =========================
# 1. 설정
# =========================
START_PAGE = 2
END_PAGE = 5
MEMBER_TAG = "LSB"   # 조원별 이니셜
OUTPUT_DIR = Path("./data/input")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://viewership.softc.one/ranking/virtualsoftcone"
    "?startDateTime=2026-02-28T15%3A00%3A00.000Z"
    "&endDateTime=2026-03-31T14%3A59%3A59.999Z"
    "&page={page}"
)

# =========================
# 2. 유틸 함수
# =========================
def wait_for_page_ready(driver, timeout=20):
    wait = WebDriverWait(driver, timeout)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    time.sleep(2)

def parse_platform(platform_key: str) -> str:
    if platform_key == "naverchzzk":
        return "CHZZK"
    elif platform_key == "afreeca":
        return "SOOP"
    return platform_key.upper()

# =========================
# 3. 페이지별 수집
# =========================
results = []
seen = set()

for page in range(START_PAGE, END_PAGE + 1):
    url = BASE_URL.format(page=page)
    print("=" * 80)
    print(f"페이지 이동: {page}")
    print(url)

    driver.get(url)
    wait_for_page_ready(driver)

    anchors = driver.find_elements(By.TAG_NAME, "a")
    page_count = 0

    for a in anchors:
        href = a.get_attribute("href")
        text = a.text.strip()

        if href and "/channel/" in href:
            m = re.search(r"/channel/([^/]+)/([^/?#]+)", href)
            if not m:
                continue

            platform_key = m.group(1)
            channel_id = m.group(2)

            # 중복 방지
            if channel_id in seen:
                continue
            seen.add(channel_id)

            lines = [line.strip() for line in text.split("\n") if line.strip()]

            rank = lines[0] if len(lines) > 0 else ""
            streamer_name = lines[1] if len(lines) > 1 else (lines[0] if lines else "")

            results.append({
                "page": page,
                "rank": rank,
                "streamer_name": streamer_name,
                "platform": parse_platform(platform_key),
                "platform_key": platform_key,
                "channel_id": channel_id,
                "detail_url": href
            })
            page_count += 1

    print(f"페이지 {page} 수집 완료: {page_count}명")

# =========================
# 4. 저장
# =========================
output_file = OUTPUT_DIR / f"{MEMBER_TAG}_streamers_p{START_PAGE}_p{END_PAGE}.csv"

with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "page",
            "rank",
            "streamer_name",
            "platform",
            "platform_key",
            "channel_id",
            "detail_url"
        ]
    )
    writer.writeheader()
    writer.writerows(results)

print("=" * 80)
print("저장 완료:", output_file)
print("총 수집 개수:", len(results))

페이지 이동: 2
https://viewership.softc.one/ranking/virtualsoftcone?startDateTime=2026-02-28T15%3A00%3A00.000Z&endDateTime=2026-03-31T14%3A59%3A59.999Z&page=2
페이지 2 수집 완료: 100명
페이지 이동: 3
https://viewership.softc.one/ranking/virtualsoftcone?startDateTime=2026-02-28T15%3A00%3A00.000Z&endDateTime=2026-03-31T14%3A59%3A59.999Z&page=3
페이지 3 수집 완료: 100명
페이지 이동: 4
https://viewership.softc.one/ranking/virtualsoftcone?startDateTime=2026-02-28T15%3A00%3A00.000Z&endDateTime=2026-03-31T14%3A59%3A59.999Z&page=4
페이지 4 수집 완료: 100명
페이지 이동: 5
https://viewership.softc.one/ranking/virtualsoftcone?startDateTime=2026-02-28T15%3A00%3A00.000Z&endDateTime=2026-03-31T14%3A59%3A59.999Z&page=5
페이지 5 수집 완료: 100명
저장 완료: data\input\LSB_streamers_p2_p5.csv
총 수집 개수: 400


[3. 스트리머별 통계 csv 내보내기 자동화]
- 기본은 바로 위 코드로 만들어진 csv로 돌리지만
- 중간에 보안 검문으로 브라우저가 끊겨서 실패한 데이터들은 failed_streamers0.csv로 따로 저장

In [6]:
import csv
import time
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# =========================
# 0. Edge 디버그 연결
# =========================
options = Options()
options.use_chromium = True
options.debugger_address = "127.0.0.1:9222"

driver = webdriver.Edge(options=options)

# =========================
# 1. 설정
# =========================
INPUT_CSV = "./data/logs/LSB_failed_streamers_retry2.csv"
MEMBER_TAG = "LSB"

DOWNLOAD_DIR = Path.home() / "Downloads"
RAW_DIR = Path("./data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DT = "2024-12-31T15%3A00%3A00.000Z"
END_DT   = "2026-03-31T14%3A59%3A59.999Z"

FAILED_OUTPUT = Path(f"./data/logs/{MEMBER_TAG}_failed_streamers.csv")
FAILED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# =========================
# 2. 유틸 함수
# =========================
def make_statistics_url(platform_key: str, channel_id: str) -> str:
    return (
        f"https://viewership.softc.one/channel/{platform_key}/{channel_id}/statistics"
        f"?startDateTime={START_DT}&endDateTime={END_DT}"
    )

def safe_name(text: str) -> str:
    bad = '<>:"/\\|?*'
    for ch in bad:
        text = text.replace(ch, "_")
    return text.strip()

def wait_for_body(driver, timeout=20):
    wait = WebDriverWait(driver, timeout)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    time.sleep(2)

def wait_for_statistics_table_ready(driver, timeout=25):
    wait = WebDriverWait(driver, timeout)
    candidates = [
        (By.XPATH, "//*[contains(., '날짜')]"),
        (By.XPATH, "//*[contains(., '평균시청자')]"),
        (By.XPATH, "//*[contains(., '최고시청자')]"),
        (By.XPATH, "//*[contains(., 'CSV 다운로드')]"),
    ]

    last_error = None
    for by, locator in candidates:
        try:
            wait.until(EC.presence_of_element_located((by, locator)))
            time.sleep(3)
            return
        except Exception as e:
            last_error = e

    if last_error:
        raise last_error

def wait_for_csv_button(driver, timeout=25):
    wait = WebDriverWait(driver, timeout)
    return wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//button[contains(., 'CSV 다운로드')]")
        )
    )

def wait_for_download_complete(download_dir: Path, before_files: set, timeout=90):
    start = time.time()

    while time.time() - start < timeout:
        current_paths = [p for p in download_dir.iterdir() if p.is_file()]
        current_names = {p.name for p in current_paths}
        new_files = current_names - before_files

        if new_files:
            completed = [
                name for name in new_files
                if not name.endswith(".crdownload") and not name.endswith(".tmp")
            ]
            if completed:
                latest_name = max(
                    completed,
                    key=lambda name: (download_dir / name).stat().st_mtime
                )
                return download_dir / latest_name

        time.sleep(1)

    return None

def is_security_checkpoint(driver) -> bool:
    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text
        keywords = [
            "브라우저를 확인하지 못했습니다",
            "보안 검문소",
            "Vercel",
            "코드 21",
        ]
        return any(k in body_text for k in keywords)
    except Exception:
        return False

# =========================
# 3. 대상 읽기
# =========================
with open(INPUT_CSV, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"대상 수: {len(rows)}")
failed_rows = []

# =========================
# 4. 실행
# =========================
for i, row in enumerate(rows, start=1):
    streamer_name = row["streamer_name"]
    platform = row["platform"]
    platform_key = row["platform_key"]
    channel_id = row["channel_id"]
    page = row.get("page", "")

    url = make_statistics_url(platform_key, channel_id)

    print("=" * 80)
    print(f"[{i}/{len(rows)}] 이동 시작: {streamer_name} (page={page})")

    try:
        driver.get(url)
        wait_for_body(driver)

        if is_security_checkpoint(driver):
            raise Exception("보안 검문소 감지")

        wait_for_statistics_table_ready(driver)

        before_files = {p.name for p in DOWNLOAD_DIR.iterdir() if p.is_file()}
        btn = wait_for_csv_button(driver)

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", btn)

        downloaded = wait_for_download_complete(DOWNLOAD_DIR, before_files, timeout=90)

        if downloaded:
            new_name = RAW_DIR / f"{safe_name(platform)}_{safe_name(streamer_name)}_{channel_id}_statistics.csv"

            if new_name.exists():
                new_name.unlink()

            downloaded.rename(new_name)
            print(f"저장 완료: {new_name.name}")
        else:
            raise Exception("다운로드 파일 감지 실패")

        time.sleep(2)

    except Exception as e:
        print(f"실패: {streamer_name} / {type(e).__name__} / {e}")
        failed_rows.append({
            "page": page,
            "rank": row.get("rank", ""),
            "streamer_name": streamer_name,
            "platform": platform,
            "platform_key": platform_key,
            "channel_id": channel_id,
            "detail_url": row.get("detail_url", ""),
            "error_type": type(e).__name__,
            "error_message": str(e)
        })

# =========================
# 5. 실패 목록 저장
# =========================
if failed_rows:
    with open(FAILED_OUTPUT, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "page",
                "rank",
                "streamer_name",
                "platform",
                "platform_key",
                "channel_id",
                "detail_url",
                "error_type",
                "error_message"
            ]
        )
        writer.writeheader()
        writer.writerows(failed_rows)

    print(f"실패 목록 저장 완료: {FAILED_OUTPUT} ({len(failed_rows)}건)")
else:
    print("실패한 대상 없음")

대상 수: 19
[1/19] 이동 시작: 해득이 (page=5)
저장 완료: CHZZK_해득이_659dcc02fa42de17387db4c8d722f55c_statistics.csv
[2/19] 이동 시작: 푸마고치 (page=5)
저장 완료: SOOP_푸마고치_fumagochi_statistics.csv
[3/19] 이동 시작: R리노아 (page=5)
저장 완료: CHZZK_R리노아_2f1107218a8e549f7b9ec5a53c488d33_statistics.csv
[4/19] 이동 시작: 엄히로 (page=5)
저장 완료: CHZZK_엄히로_60761dc68dca3092c55c4371710ce848_statistics.csv
[5/19] 이동 시작: 묭 이 (page=5)
저장 완료: CHZZK_묭 이_58bb93cf187737941c1a84cace6a86c4_statistics.csv
[6/19] 이동 시작: 린코♥ (page=5)
저장 완료: SOOP_린코♥_linco00_statistics.csv
[7/19] 이동 시작: 씹은성 (page=5)
저장 완료: SOOP_씹은성_luxury1004_statistics.csv
[8/19] 이동 시작: 서피카 (page=5)
저장 완료: SOOP_서피카_spica21_statistics.csv
[9/19] 이동 시작: 미 유 (page=5)
저장 완료: CHZZK_미 유_d58ace43901fd369f1b4efa4ed175103_statistics.csv
[10/19] 이동 시작: 미사키 하루 (page=5)
저장 완료: CHZZK_미사키 하루_9f7467f5f3dfa4ea3dcef2962187d6a2_statistics.csv
[11/19] 이동 시작: 도화령 (page=5)
저장 완료: CHZZK_도화령_d7b5e8a2bd0bb3202201fa707a726f80_statistics.csv
[12/19] 이동 시작: 찐녕 (page=5)
저장 완료: SOOP_찐녕_wlssud44_statistics.csv


[4. 스트리머별 통계 csv 병합]
- 병합하면서 streamer_name, platform 컬럼을 메타 데이터로 추출
- 병합한 데이터는 ./data/processed로 기록

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("./data/raw")
OUTPUT_PATH = Path("./data/processed/final_streamer_data.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

dfs = []

files = list(RAW_DIR.glob("*_statistics.csv"))
print("찾은 파일 수:", len(files))

for file in files:
    filename = file.stem  # .csv 제거된 이름
    print("처리 중:", file.name)

    try:
        # 뒤에서부터 자르기
        base = filename[:-11]  # "_statistics" 제거
        platform, rest = base.split("_", 1)
        streamer_name, channel_id = rest.rsplit("_", 1)

        df = pd.read_csv(file)
        df["streamer_name"] = streamer_name
        df["platform"] = platform
        df["channel_id"] = channel_id

        dfs.append(df)

    except Exception as e:
        print("에러:", file.name, e)

if not dfs:
    raise ValueError("병합할 데이터프레임이 없습니다.")

final_df = pd.concat(dfs, ignore_index=True)
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("=" * 50)
print("통합 완료")
print("shape:", final_df.shape)
print("저장 위치:", OUTPUT_PATH)

찾은 파일 수: 500
처리 중: CHZZK_404 덩기덕_41e6c86cf804123cc8574b4cd118c019_statistics.csv
처리 중: CHZZK_HeavyRainy_aef0c98592271de9ec4906eca2987e05_statistics.csv
처리 중: CHZZK_RED레드_a96cea2d2c39cec636ba8170c66a0510_statistics.csv
처리 중: CHZZK_R로제타_faf9cdd37e2ce5e2af6944dbf42739a9_statistics.csv
처리 중: CHZZK_R리노아_2f1107218a8e549f7b9ec5a53c488d33_statistics.csv
처리 중: CHZZK_ㅇ연이ㅇ_75b643e50481d4ec373f77b58a41746f_statistics.csv
처리 중: CHZZK_ㅎㄹㄹㅎㄹㄹ_733ac918d624f6034f8786efc318e433_statistics.csv
처리 중: CHZZK_감군장입니다_ba1469a50bdc7f420d775f43f29b1414_statistics.csv
처리 중: CHZZK_감제이님_e8da946c7a7c10d8de959cff09e306c6_statistics.csv
처리 중: CHZZK_갓단미_7dc9561a4e60fb5221bdea64346672f1_statistics.csv
처리 중: CHZZK_강다림_5ebc36275f94b1342ef77e8ca5302d87_statistics.csv
처리 중: CHZZK_강단해_ae21a2b6fd79853e78973b8825b1e3cf_statistics.csv
처리 중: CHZZK_강지_b5ed5db484d04faf4d150aedd362f34b_statistics.csv
처리 중: CHZZK_계춘회_a9a343510e132ea3026ff3cf682820b5_statistics.csv
처리 중: CHZZK_고뇽찌_78d950b1c85bffff10367c4735aa6cfc_statistics.csv
처리 중:

NameError: name 'file_size_mb' is not defined

In [8]:
# ✅ 파일 크기 확인 (MB 단위)
file_size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
print(f"파일 크기: {file_size_mb:.2f} MB")

파일 크기: 13.65 MB
